<a href="https://colab.research.google.com/github/mzaib1012/ieee14-load-flow-analysis/blob/main/python-pypsa/IEEE14_Static_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install PyPSA
!pip install pypsa

import pypsa
import pandas as pd
import numpy as np

# 2. Initialize the Network
network = pypsa.Network()
base_mva = 100.0 # MVA Base
network.set_snapshots([2026])

# 3. Add All 14 Buses with their Target Voltage Setpoints
v_mag_pu_init = [1.06, 1.045, 1.01, 1.0, 1.0, 1.07, 1.0, 1.09, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
for i in range(1, 15):
    network.add("Bus", f"Bus {i}", v_nom=132.0, v_mag_pu_set=v_mag_pu_init[i-1])

# 4. Add Generators (Explicitly providing a PV/Slack control environment)
network.add("Generator", "Gen 1 (Slack)", bus="Bus 1", control="Slack", p_set=0.0)
network.add("Generator", "Gen 2", bus="Bus 2", control="PV", p_set=40.0)
network.add("Generator", "Gen 3", bus="Bus 3", control="PV", p_set=0.0)
network.add("Generator", "Gen 6", bus="Bus 6", control="PV", p_set=0.0)
network.add("Generator", "Gen 8", bus="Bus 8", control="PV", p_set=0.0)

# 5. Add All 11 Non-Zero Active/Reactive Loads
loads = {
    "Bus 2": [21.7, 12.7],  "Bus 3": [94.2, 19.0], "Bus 4": [47.8, -3.9],
    "Bus 5": [7.6, 1.6],   "Bus 6": [11.2, 7.5],  "Bus 9": [29.5, 16.6],
    "Bus 10": [9.0, 5.8],  "Bus 11": [3.5, 1.8],  "Bus 12": [6.1, 1.6],
    "Bus 13": [13.5, 5.8], "Bus 14": [14.9, 5.0]
}
for bus_id, values in loads.items():
    network.add("Load", f"Load {bus_id}", bus=bus_id, p_set=values[0], q_set=values[1])

# 6. Complete IEEE 14-Bus Line and Transformer Branch Topology Matrix
lines_data = [
    # Main Transmission Lines (Buses 1-5 Core)
    ("L1-2", "Bus 1", "Bus 2", 0.01938, 0.05917),
    ("L1-5", "Bus 1", "Bus 5", 0.05403, 0.22304),
    ("L2-3", "Bus 2", "Bus 3", 0.04699, 0.19797),
    ("L2-4", "Bus 2", "Bus 4", 0.05811, 0.17632),
    ("L2-5", "Bus 2", "Bus 5", 0.05695, 0.17388),
    ("L3-4", "Bus 3", "Bus 4", 0.06701, 0.17103),
    ("L4-5", "Bus 4", "Bus 5", 0.01335, 0.04211),

    # Transformer Branches linking Core to Outer Loop (Buses 6-14)
    ("L5-6", "Bus 5", "Bus 6", 0.00000, 0.25202),
    ("L4-7", "Bus 4", "Bus 7", 0.00000, 0.20912),
    ("L4-9", "Bus 4", "Bus 9", 0.00000, 0.55618),
    ("L7-8", "Bus 7", "Bus 8", 0.00000, 0.17615),
    ("L7-9", "Bus 7", "Bus 9", 0.00000, 0.11001),

    # Outer Distribution Network Interconnections
    ("L6-11", "Bus 6", "Bus 11", 0.09498, 0.19890),
    ("L6-12", "Bus 6", "Bus 12", 0.12291, 0.25581),
    ("L6-13", "Bus 6", "Bus 13", 0.06615, 0.13027),
    ("L9-10", "Bus 9", "Bus 10", 0.03181, 0.08450),
    ("L9-14", "Bus 9", "Bus 14", 0.12711, 0.27038),
    ("L10-11", "Bus 10", "Bus 11", 0.08205, 0.19207),
    ("L12-13", "Bus 12", "Bus 13", 0.22092, 0.19988),
    ("L13-14", "Bus 13", "Bus 14", 0.17093, 0.34802)
]

for name, b1, b2, r, x in lines_data:
    network.add("Line", name, bus0=b1, bus1=b2, r=r, x=x)

# 7. Run Full Newton-Raphson Load Flow Simulation
print("Executing Newton-Raphson power flow calculation...\n")
success = network.pf()

if success == "ok":
    print("\n✅ Power flow converged successfully!")
    print("\n--- BUS VOLTAGE PROFILES (per-unit) ---")
    print(network.buses_t.v_mag_pu.loc[2026].round(4))

    print("\n--- REAL POWER TRANSMISSION (MW per line) ---")
    print(network.lines_t.p0.loc[2026].round(2))
else:
    print("❌ Power flow failed to converge.")

Executing Newton-Raphson power flow calculation...

❌ Power flow failed to converge.
